# Match Simbad Targets with LSSTCam Sources Catalogue

For each target star (identified by `simbad_id`, `ra_deg`, `dec_deg`) read from
the input CSV, this notebook:

1. Locates the Butler tract/patch that covers the target coordinates.
2. Loads the corresponding `source` (or equivalent catalogue dataset
   auto-discovered from the registry).
3. Performs a nearest-neighbour sky cross-match using
   `astropy.coordinates.SkyCoord.match_to_catalog_sky`.
4. Retains matches within a configurable search radius.
5. Saves the merged table (targets + matched LSST object columns) to CSV/Parquet.


- refer to : https://github.com/lsst/tutorial-notebooks/blob/main/DP1/200_Data_Products/201_Catalogs/201_2_Source_table.ipynb

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-26
- **Last update:** 2026-06-26


## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

## Helper utilities

In [ ]:
def dataset_type_exists(butler, name):
    """Return True if *name* is a registered dataset type in *butler*."""
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

## Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "MATCHSRC_03"
DIR_DATA_IN = "data_DEEPCCUTOUTS_01_in"  # reuse the same input directory as NB01
DIR_DATA_OUT = f"data_{NB_TAG}_out"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
print(f"Data output: {os.path.abspath(DIR_DATA_OUT)}")
print(f"Figs       : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG inside DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

In [ ]:
# ── Butler ────────────────────────────────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

BANDSEL = "r"

# ── Input targets ─────────────────────────────────────────────────────────────
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# ── Cross-match search radius ─────────────────────────────────────────────────
MATCH_RADIUS_ARCSEC = 1.0  # maximum separation for a valid match [arcsec]

DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
time_start = Time(DATE_START, format="isot", scale="utc")
time_stop = Time(DATE_STOP, format="isot", scale="utc")
MJD_START = time_start.mjd
MJD_STOP = time_stop.mjd
DELTAMJD_DAYS = MJD_STOP - MJD_START
print(f"MJD ::: start = {MJD_START} , stop = {MJD_STOP} , delta t = {DELTAMJD_DAYS} days")

SRC_COLUMNS = [
    "coord_ra",
    "coord_dec",
    "parentSourceId",
    "x",
    "y",
    "xErr",
    "yErr",
    "ra",
    "dec",
    "raErr",
    "decErr",
    "calibFlux",
    "calibFluxErr",
    "ap03Flux",
    "ap03FluxErr",
    "ap03Flux_flag",
    "ap06Flux",
    "ap06FluxErr",
    "ap06Flux_flag",
    "ap09Flux",
    "ap09FluxErr",
    "ap09Flux_flag",
    "ap12Flux",
    "ap12FluxErr",
    "ap12Flux_flag",
    "ap17Flux",
    "ap17FluxErr",
    "ap17Flux_flag",
    "ap25Flux",
    "ap25FluxErr",
    "ap25Flux_flag",
    "ap35Flux",
    "ap35FluxErr",
    "ap35Flux_flag",
    "ap50Flux",
    "ap50FluxErr",
    "ap50Flux_flag",
    "ap70Flux",
    "ap70FluxErr",
    "ap70Flux_flag",
    "sky",
    "skyErr",
    "psfFlux",
    "psfFluxErr",
    #'ixx','iyy','ixy','ixxPSF','iyyPSF','ixyPSF','ixxDebiasedPSF','iyyDebiasedPSF','ixyDebiasedPSF',
    #'gaussianFlux','gaussianFluxErr',
    "extendedness",
    "sizeExtendedness",
    #'blendedness_abs','blendedness_flag','blendedness_flag_noCentroid','blendedness_flag_noShape',
    "apFlux_12_0_flag",
    "apFlux_12_0_flag_apertureTruncated",
    "apFlux_12_0_flag_sincCoeffsTruncated",
    "apFlux_12_0_instFlux",
    "apFlux_12_0_instFluxErr",
    "apFlux_17_0_flag",
    "apFlux_17_0_instFlux",
    "apFlux_17_0_instFluxErr",
    "apFlux_35_0_flag",
    "apFlux_35_0_instFlux",
    "apFlux_35_0_instFluxErr",
    "apFlux_50_0_flag",
    "apFlux_50_0_instFlux",
    "apFlux_50_0_instFluxErr",
    #'normCompTophatFlux_flag','normCompTophatFlux_instFlux','normCompTophatFlux_instFluxErr',
    "extendedness_flag",
    "sizeExtendedness_flag",
    #'footprintArea_value','invalidPsfFlag','jacobian_flag','jacobian_value',
    "localBackground_instFlux",
    "localBackground_instFluxErr",
    "localBackground_flag",
    "localBackground_flag_noGoodPixels",
    "localBackground_flag_noPsf",
    #'pixelFlags_bad','pixelFlags_cr','pixelFlags_crCenter','pixelFlags_edge','pixelFlags_interpolated','pixelFlags_interpolatedCenter','pixelFlags_nodata','pixelFlags_offimage','pixelFlags_saturated','pixelFlags_saturatedCenter','pixelFlags_suspect','pixelFlags_suspectCenter',
    #'psfFlux_apCorr','psfFlux_apCorrErr',
    #'psfFlux_area','psfFlux_flag','psfFlux_flag_apCorr','psfFlux_flag_edge','psfFlux_flag_noGoodPixels',
    #'gaussianFlux_flag','centroid_flag','centroid_flag_almostNoSecondDerivative','centroid_flag_badError','centroid_flag_edge','centroid_flag_noSecondDerivative','centroid_flag_notAtMaximum','centroid_flag_resetToPeak',
    #'variance_flag','variance_flag_emptyFootprint','variance_value',
    #'calib_astrometry_used','calib_photometry_reserved','calib_photometry_used','calib_psf_candidate','calib_psf_reserved','calib_psf_used',
    #'deblend_deblendedAsPsf','deblend_hasStrayFlux','deblend_masked','deblend_nChild','deblend_parentTooBig','deblend_patchedTemplate','deblend_rampedTemplate','deblend_skipped','deblend_tooManyPeaks',
    #'hsmPsfMoments_flag','hsmPsfMoments_flag_no_pixels','hsmPsfMoments_flag_not_contained','hsmPsfMoments_flag_parent_source',
    #'iDebiasedPSF_flag','iDebiasedPSF_flag_no_pixels','iDebiasedPSF_flag_not_contained','iDebiasedPSF_flag_parent_source','iDebiasedPSF_flag_galsim','iDebiasedPSF_flag_edge',
    #'hsmShapeRegauss_flag','hsmShapeRegauss_flag_galsim','hsmShapeRegauss_flag_no_pixels','hsmShapeRegauss_flag_not_contained','hsmShapeRegauss_flag_parent_source',
    "sky_source",
    "visit",
    "detector",
    "band",
    "physical_filter",
    "sourceId",
]

print("Butler configuration done.")

## Load targets

In [ ]:
df_targets = pd.read_csv(os.path.join(DIR_DATA_IN, target_file))
print(f"Loaded {len(df_targets)} targets from {target_file}")
display(df_targets.head())

## Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
print(f"Butler initialised | repo: {repo}")

## Auto-discover the object-table dataset type

The catalogue dataset name changed across pipeline versions:

| Pipeline era | Dataset type name          |
|:-------------|:---------------------------|
| Gen2 / HSC   | `deepCoadd_obj`            |
| DP1          | `objectTable_tract`        |
| DP2+         | `object_table_tract`       |

We probe the registry so the notebook is collection-agnostic.


In [ ]:
# Prioritised list of candidate object-table dataset type names
SRC_TABLE_CANDIDATES = [
    "source",
    "sourceTable," "sourceTable_visit",  # visit-level source table (fallback)
]

# List all source-table-related types actually in the registry
all_src_types = [
    d.name for d in registry.queryDatasetTypes() if "source" in d.name.lower() or "table" in d.name.lower()
]
print("src-table-related dataset types in registry:")
# for t in sorted(all_src_types):
#    print(f"  {t}")

# Pick the first candidate that is registered
SRC_DATASET = None
for name in SRC_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        SRC_DATASET = name
        print(f"\n✔ Selected src-table dataset type: '{SRC_DATASET}'")
        break

if SRC_DATASET is None:
    raise RuntimeError(
        "No recognised source-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_src_types}"
    )

## Inspect the schema of the object table (first available tract)

We load a single object table to check which columns are available before
the main loop.

In [ ]:
# Use the coordinates of the first target to locate a representative tract
first_row = df_targets.iloc[0]


target_name = first_row["simbad_id"]
target_field = first_row["field"]
target_ra = first_row["ra_deg"]
target_dec = first_row["dec_deg"]


probe_point = SpherePoint(target_ra, target_dec, degrees)

probe_tract_info = skymap.findTract(probe_point)
probe_patch_info = probe_tract_info.findPatch(probe_point)

probe_tract_id = probe_tract_info.getId()
probe_patch_id = probe_patch_info.getSequentialIndex()

probe_dataId = {"tract": probe_tract_id, "patch": probe_patch_id, "skymap": skymapName}


#
time1 = Time(MJD_START, format="mjd", scale="tai")
time2 = Time(MJD_STOP, format="mjd", scale="tai")
timespan = Timespan(time1, time2)

refs = butler.query_datasets(
    "source",
    where="visit.timespan OVERLAPS :timespan AND \
                             visit_detector_region.region OVERLAPS POINT(:ra, :dec)",
    bind={"timespan": timespan, "ra": target_ra, "dec": target_dec},
)
# number or light-curve points == ( number of visits) including all bands : ex Number of points 636
print("Number of points", len(refs))
# example Type of source (instrument, visit, band, day_obs,physical_filter {instrument: 'LSSTCam', visit: 2025051500122, band: 'i', day_obs: 20250515, physical_filter: 'i_39'}
print("Type of source (instrument, visit, band, day_obs,physical_filter", refs[0].dataId)
# Now really fetch the visits
results = butler.get(refs[0])

In [ ]:
print("Number of source", len(results))
print(results.columns)

In [ ]:
# Identify the RA/Dec and objectId column names from the schema
# They can vary between pipeline versions.


def find_col(df, candidates):
    """Return the first column name from *candidates* that exists in *df*."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found in table columns: {list(df.columns[:20])}")


RA_COL = find_col(results, ["coord_ra", "ra", "RA", "ra_deg"])
DEC_COL = find_col(results, ["coord_dec", "dec", "DEC", "dec_deg"])
ID_COL = find_col(results, ["objectId", "object_id", "id", "sourceId"])

print(f"RA column  : {RA_COL}")
print(f"Dec column : {DEC_COL}")
print(f"ID column  : {ID_COL}")

## Main cross-match loop over sources bunches (refs)

For every target we:
1. Find the tract that contains the target.
2. Load `objectTable_tract` for that tract (cached: each tract is loaded only once).
3. Cross-match with `SkyCoord.match_to_catalog_sky`.
4. Keep the match only if separation ≤ `MATCH_RADIUS_ARCSEC`.


In [ ]:
# Cache: avoid reloading the same patch table multiple times.
# Key: (tract_id, patch_id)  — patch_id is the integer index returned by
# tract_info.findPatch(point).getId().getSequential()
patch_cache: dict[tuple[int, int], pd.DataFrame] = {}


def get_object_table_patch(tract_id: int, patch_id: int) -> pd.DataFrame:
    """Load (and cache) the object table for a single *patch* of *tract_id*.

    Loading by patch instead of the full tract dramatically reduces memory
    consumption on the RSP (a typical DP2 tract has ~1 M objects; a single
    patch holds ~5-20 k objects).
    """
    key = (tract_id, patch_id)
    if key in patch_cache:
        return patch_cache[key]

    dataId = {"tract": tract_id, "patch": patch_id, "skymap": skymapName}
    print(
        f"  Loading object table for tract {tract_id} / patch {patch_id} … ",
        end="",
        flush=True,
    )
    try:
        df = butler.get(OBJ_DATASET, dataId=dataId)
        if not isinstance(df, pd.DataFrame):
            try:
                df = df.to_pandas()
            except AttributeError:
                df = pd.DataFrame(df)
        print(f"{len(df):,} objects")
        patch_cache[key] = df
        return df
    except Exception as exc:
        print(f"FAILED ({exc})")
        return None


print("Patch cache initialised.")

In [ ]:
results = []  # list of dicts; one per target

for idx, target in df_targets.iterrows():
    simbad_id = target["simbad_id"]
    ra_t = target["ra_deg"]
    dec_t = target["dec_deg"]

    # ── 1.  Find tract **and patch** ─────────────────────────────────────────
    point = SpherePoint(ra_t, dec_t, degrees)
    tract_info = skymap.findTract(point)
    tract_id = tract_info.getId()

    patch_info = tract_info.findPatch(point)
    patch_id = patch_info.sequential_index  # integer patch index

    print(
        f"[{idx:3d}] {simbad_id:40s}  ra={ra_t:.5f}  dec={dec_t:+.5f}" f"  tract={tract_id}  patch={patch_id}"
    )

    # ── 2.  Load object table for this patch only (cached) ───────────────────
    df_obj = get_object_table_patch(tract_id, patch_id)

    if df_obj is None or len(df_obj) == 0:
        print(f"       *** no object table available for tract {tract_id} / patch {patch_id} — skipping")
        row = target.to_dict()
        row.update(
            {
                "lsst_objectId": None,
                "lsst_ra": None,
                "lsst_dec": None,
                "separation_arcsec": None,
                "match_status": "no_table",
                "lsst_tract": tract_id,
                "lsst_patch": patch_id,
            }
        )
        results.append(row)
        continue

    # ── 3.  Build SkyCoord catalogue ─────────────────────────────────────────
    ra_obj = df_obj[RA_COL].values
    dec_obj = df_obj[DEC_COL].values

    if ra_obj.max() <= 2 * np.pi + 0.1:  # heuristic: radians
        unit_sky = u.rad
    else:  # degrees
        unit_sky = u.deg

    cat_sky = SkyCoord(ra=ra_obj * unit_sky, dec=dec_obj * unit_sky)
    tgt_sky = SkyCoord(ra=ra_t * u.deg, dec=dec_t * u.deg)

    # ── 4.  Nearest-neighbour match ──────────────────────────────────────────
    best_idx, sep2d, _ = tgt_sky.match_to_catalog_sky(cat_sky)
    sep_arcsec = sep2d.to(u.arcsec).value

    matched_obj = df_obj.iloc[best_idx]

    row = target.to_dict()
    row["lsst_tract"] = tract_id
    row["lsst_patch"] = patch_id

    if sep_arcsec <= MATCH_RADIUS_ARCSEC:
        lsst_ra = matched_obj[RA_COL]
        lsst_dec = matched_obj[DEC_COL]
        if unit_sky == u.rad:
            lsst_ra = np.degrees(lsst_ra)
            lsst_dec = np.degrees(lsst_dec)

        row.update(
            {
                "lsst_objectId": int(matched_obj[ID_COL]),
                "lsst_ra": lsst_ra,
                "lsst_dec": lsst_dec,
                "separation_arcsec": sep_arcsec,
                "match_status": "matched",
            }
        )
        print(f"       \u2714 matched  objectId={matched_obj[ID_COL]}  sep={sep_arcsec:.3f}")
    else:
        row.update(
            {
                "lsst_objectId": None,
                "lsst_ra": None,
                "lsst_dec": None,
                "separation_arcsec": sep_arcsec,
                "match_status": "no_match",
            }
        )
        print(f"       \u2718 no match  (closest sep={sep_arcsec:.3f} arcsec > {MATCH_RADIUS_ARCSEC} arcsec)")

    results.append(row)

print(f"\nDone: {len(results)} targets processed.")

## Results summary

In [ ]:
df_results = pd.DataFrame(results)

n_total = len(df_results)
n_matched = (df_results["match_status"] == "matched").sum()
n_nomatch = (df_results["match_status"] == "no_match").sum()
n_notable = (df_results["match_status"] == "no_table").sum()

print(f"Total targets : {n_total}")
print(f"  Matched     : {n_matched}  ({100*n_matched/n_total:.1f}%)")
print(f"  No match    : {n_nomatch}  (sep > {MATCH_RADIUS_ARCSEC} arcsec)")
print(f"  No table    : {n_notable}  (Butler error)")

display(df_results)

## Diagnostic plots

In [ ]:
df_matched = df_results[df_results["match_status"] == "matched"].copy()

# ── Distribution of separations ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(df_matched["separation_arcsec"], bins=20, edgecolor="k", linewidth=0.5)
ax.axvline(MATCH_RADIUS_ARCSEC, color="red", linestyle="--", label=f'search radius = {MATCH_RADIUS_ARCSEC}"')
ax.set_xlabel("Separation (arcsec)")
ax.set_ylabel("Number of targets")
ax.set_title("Cross-match separation — Simbad targets vs LSST objects")
ax.legend()
plt.tight_layout()
savefig("crossmatch_separation_histogram")
plt.show()

In [ ]:
# ── RA/Dec offset (target − LSST object) ─────────────────────────────────────
cos_dec = np.cos(np.radians(df_matched["dec_deg"].values))
d_ra = (df_matched["ra_deg"].values - df_matched["lsst_ra"].values) * cos_dec * 3600  # arcsec
d_dec = (df_matched["dec_deg"].values - df_matched["lsst_dec"].values) * 3600  # arcsec

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(d_ra, d_dec, s=20, alpha=0.7)
circle = plt.Circle(
    (0, 0), MATCH_RADIUS_ARCSEC, color="red", fill=False, linestyle="--", label=f'r = {MATCH_RADIUS_ARCSEC}"'
)
ax.add_patch(circle)
ax.axhline(0, color="k", linewidth=0.5)
ax.axvline(0, color="k", linewidth=0.5)
ax.set_xlabel(r"$\Delta\,\alpha\cos\delta$ (arcsec)")
ax.set_ylabel(r"$\Delta\,\delta$ (arcsec)")
ax.set_title("Positional offsets: Simbad − LSST")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
savefig("crossmatch_radec_offsets")
plt.show()

In [ ]:
# ── Summary pie chart ─────────────────────────────────────────────────────────
counts = df_results["match_status"].value_counts()
fig, ax = plt.subplots(figsize=(4, 4))
ax.pie(counts.values, labels=counts.index, autopct="%1.0f%%", startangle=90)
ax.set_title(f'Match status (r < {MATCH_RADIUS_ARCSEC}" )')
plt.tight_layout()
savefig("crossmatch_status_pie")
plt.show()

## Save results

In [ ]:
out_stem = os.path.join(DIR_DATA_OUT, "targets_matched_lsst_objects")

df_results.to_csv(out_stem + ".csv", index=False)
df_results.to_parquet(out_stem + ".parquet", index=False)

print(f"Full results  → {out_stem}.csv  /  .parquet")
print(f"  {len(df_results)} rows total, {n_matched} matched")

# Save the matched-only subset for downstream notebooks
out_matched = os.path.join(DIR_DATA_OUT, "targets_matched_only")
df_matched.to_csv(out_matched + ".csv", index=False)
df_matched.to_parquet(out_matched + ".parquet", index=False)
print(f"Matched only  → {out_matched}.csv  /  .parquet")

## Additional columns from the LSST object table

The cell below enriches the matched table with extra LSST photometric columns
(PSF flux, extended-ness classifier, …) that may be useful in downstream
notebooks.


In [ ]:
# Columns to transfer from the LSST object table to the matched result
# (filter to those actually present in the schema discovered above)
EXTRA_COLS_WANTED = [
    # Photometry
    "r_psfFlux",
    "r_psfFluxErr",
    "g_psfFlux",
    "g_psfFluxErr",
    "i_psfFlux",
    "i_psfFluxErr",
    # Morphology / star-galaxy separator
    "r_extendedness",
    "r_sersicIndex",
    # Shape
    "r_ixx",
    "r_iyy",
    "r_ixy",
    # Flags
    "detect_isPrimary",
    "r_pixelFlags_saturated",
    "r_pixelFlags_cr",
]

# Restrict to columns that actually exist in the probe table
extra_cols = [c for c in EXTRA_COLS_WANTED if c in df_probe.columns]
print(f"Extra columns found in schema ({len(extra_cols)}/{len(EXTRA_COLS_WANTED)}):")
print("  ", extra_cols)

if extra_cols:
    # Build a look-up DataFrame keyed by objectId
    enriched_rows = []

    for tract_id, df_obj in tract_cache.items():
        sub = df_results[
            (df_results["lsst_tract"] == tract_id) & (df_results["match_status"] == "matched")
        ].copy()
        if len(sub) == 0:
            continue

        obj_ids = sub["lsst_objectId"].astype(int).values
        cols_to_get = [ID_COL] + extra_cols
        cols_present = [c for c in cols_to_get if c in df_obj.columns]

        obj_sub = df_obj.set_index(ID_COL)[extra_cols]
        sub = sub.set_index("lsst_objectId").join(obj_sub, how="left").reset_index()
        enriched_rows.append(sub)

    if enriched_rows:
        df_enriched = pd.concat(enriched_rows, ignore_index=True)

        out_enriched = os.path.join(DIR_DATA_OUT, "targets_matched_enriched")
        df_enriched.to_csv(out_enriched + ".csv", index=False)
        df_enriched.to_parquet(out_enriched + ".parquet", index=False)
        print(f"Enriched table → {out_enriched}.csv  /  .parquet")
        display(df_enriched.head())
    else:
        print("No enriched rows to save.")
else:
    print("None of the requested extra columns were found — skipping enrichment.")